# cra-scraper — CRA Exam Ratings Scraper & Analyzer
## Demo: Analyzing Community Reinvestment Act Exam Data

This notebook demonstrates how to use cra-scraper to:
- Search the FFIEC CRA ratings database programmatically
- Compute rating distributions across regulators and states
- Identify banks with low CRA ratings (CDFI partnership opportunities)
- Track rating history and detect downgrades over time
- Parse Performance Evaluation PDFs into structured data
- Generate full Markdown summary reports

Data source: FFIEC CRA Ratings Database — free, no API key required.
Coverage: All FDIC-insured institutions examined under the Community Reinvestment Act.


In [ ]:
import sys
sys.path.insert(0, '..')

from crascraper import (
    search_ratings, parse_pe_text,
    to_dataframe, rating_distribution, rating_by_state, rating_by_regulator,
    find_partnership_opportunities, build_history, find_downgrades,
    summary_report,
    CRA_RATINGS, RATING_SCORES, REGULATORS, EXAM_METHODS,
)
from crascraper.scrapers.ffiec import _sample_ratings
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

print("cra-scraper loaded successfully")
print(f"\nCRA rating categories: {list(CRA_RATINGS.keys())}")
print(f"Federal regulators:    {list(REGULATORS.keys())}")
print(f"Exam methods:          {list(EXAM_METHODS.keys())}")


## 1. Load CRA Rating Data

We use sample data here that reflects realistic distributions across regulators
and rating categories. To pull live data from FFIEC:

    ratings = search_ratings(bank_name="Federal", state="CA")


In [ ]:
# Use sample data for demo (no scraping required)
ratings = _sample_ratings()

print(f"Loaded {len(ratings)} CRA ratings\n")
for r in ratings:
    assets = f"${r.asset_size_mm:,.0f}MM" if r.asset_size_mm else "N/A"
    print(f"  {r.bank_name:<35} {r.rating:<28} {r.exam_date}  ({r.state}, {assets})")


## 2. Rating Distribution

What's the overall distribution of CRA ratings across the dataset?


In [ ]:
dist = rating_distribution(ratings)
print("CRA Rating Distribution:")
print("=" * 55)
for _, row in dist.iterrows():
    bar = "█" * int(row['pct'] / 5)
    print(f"  {row['rating']:<30} {row['count']:>3}  {row['pct']:5.1f}%  {bar}")

print(f"\nTotal exams: {dist['count'].sum()}")


## 3. Rating Distribution by Regulator

Different regulators may show different rating patterns based on the
institutions they supervise.


In [ ]:
by_reg = rating_by_regulator(ratings)
print("Rating Performance by Regulator:")
print("=" * 60)
print(by_reg.to_string(index=False))


## 4. Rating Distribution by State

In [ ]:
by_state = rating_by_state(ratings)
print("Rating Performance by State:")
print("=" * 60)
print(by_state.to_string(index=False))


## 5. Identify CDFI Partnership Opportunities

Banks with "Needs to Improve" or "Substantial Noncompliance" ratings face
regulatory pressure to demonstrate improved community lending. These banks
often look for CDFI partnerships, community development loans, and impact
investments to improve their next CRA evaluation.


In [ ]:
opportunities = find_partnership_opportunities(ratings)
print(f"Found {len(opportunities)} banks flagged as partnership opportunities\n")

if len(opportunities) > 0:
    print(opportunities[["bank_name", "city", "state", "rating",
                          "exam_date", "asset_size_mm"]].to_string(index=False))


## 6. Filter by State or Asset Size

Find partnership opportunities in specific markets or asset size brackets.


In [ ]:
# Banks under $500MM in Illinois with low ratings
midwest_opps = find_partnership_opportunities(
    ratings,
    max_assets=500_000_000,
)

print("Partnership opportunities at banks under $500MM:")
print("=" * 65)
if len(midwest_opps) > 0:
    print(midwest_opps[["bank_name", "city", "state", "rating",
                         "asset_size_mm"]].to_string(index=False))
else:
    print("No matching opportunities in the sample data.")


## 7. Track Rating History Over Time

For banks with multiple exams over time, track how their rating has evolved.


In [ ]:
from crascraper.data.schema import CRARating

# Build a multi-exam history for a hypothetical bank
history_data = [
    CRARating(bank_id="55555", bank_name="Sample Trust Bank",
              rating="Outstanding", exam_date="2018-06-01", asset_size=300_000_000),
    CRARating(bank_id="55555", bank_name="Sample Trust Bank",
              rating="Satisfactory", exam_date="2021-06-15", asset_size=350_000_000),
    CRARating(bank_id="55555", bank_name="Sample Trust Bank",
              rating="Needs to Improve", exam_date="2024-04-20", asset_size=400_000_000),
]

histories = build_history(history_data)
for bank_id, hist in histories.items():
    print(f"\nBank: {hist.bank_name}")
    print(f"Total exams: {hist.count}")
    print(f"Latest rating: {hist.latest_rating.rating} ({hist.latest_rating.exam_date})")
    print(f"Trend: {hist.trend}")
    print(f"Has downgrade: {hist.has_downgrade}")
    print()
    print("Exam History:")
    for r in sorted(hist.ratings, key=lambda x: x.exam_date):
        print(f"  {r.exam_date}: {r.rating} (score: {r.rating_score})")


## 8. Find Banks with Downgrades

Banks that have been downgraded between exams are particularly attractive
CDFI partnership targets — they're under regulatory pressure to improve
before their next examination.


In [ ]:
downgrades = find_downgrades(history_data)

if len(downgrades) > 0:
    print("Banks with rating downgrades:")
    print("=" * 75)
    print(downgrades.to_string(index=False))
else:
    print("No downgrades detected in this dataset.")


## 9. Parse Performance Evaluation PDFs

Federal regulators publish detailed Performance Evaluation reports as PDFs.
The parser extracts structured data from these reports.


In [ ]:
sample_pe_text = '''
PUBLIC DISCLOSURE

January 15, 2024

COMMUNITY REINVESTMENT ACT
PERFORMANCE EVALUATION

Sample Community Bank
Charter Number: 12345

Office of the Comptroller of the Currency

INSTITUTION'S CRA RATING: "Satisfactory"

The Lending Test is rated: High Satisfactory
The Investment Test is rated: Outstanding
The Service Test is rated: Low Satisfactory

ASSESSMENT AREAS:
The bank's assessment areas include:
- Chicago-Naperville-Elgin MSA
- Milwaukee-Waukesha MSA

CONCLUSIONS WITH RESPECT TO PERFORMANCE TESTS

The bank's performance is...
'''

pe = parse_pe_text(sample_pe_text)
print(f"Parsed Performance Evaluation:")
print(f"  Bank Name:          {pe.bank_name}")
print(f"  Charter Number:     {pe.bank_id}")
print(f"  Exam Date:          {pe.exam_date}")
print(f"  Regulator:          {pe.regulator}")
print(f"  Overall Rating:     {pe.overall_rating}")
print(f"  Lending Test:       {pe.lending_test_rating}")
print(f"  Investment Test:    {pe.investment_test_rating}")
print(f"  Service Test:       {pe.service_test_rating}")
print(f"\n  Assessment Areas:")
for area in pe.assessment_areas:
    print(f"    - {area}")


## 10. Generate Full Summary Report

In [ ]:
report = summary_report(ratings)
print(report)


## 11. Live FFIEC API Usage

To search live FFIEC ratings:


In [ ]:
# Uncomment to run live FFIEC searches:

# Search by bank name
# results = search_ratings(bank_name="Federal")

# Search by state
# results = search_ratings(state="IL")

# Search by bank ID (FDIC certificate or OCC charter)
# results = search_ratings(bank_id="57542")

print("FFIEC CRA Ratings Database:")
print("  URL: https://www.ffiec.gov/craratings/")
print("  Free, no API key required")
print()
print("Search live data with:")
print("  from crascraper import search_ratings")
print("  results = search_ratings(bank_name='Federal', state='CA')")


## Summary

This notebook demonstrated the full cra-scraper workflow:

1. **Load CRA rating data** — sample or live FFIEC API
2. **Rating distribution** — overall and by regulator/state
3. **Partnership opportunities** — banks needing community lending improvement
4. **Filter by market** — state and asset size segmentation
5. **Rating history** — track changes over multiple exam periods
6. **Downgrade detection** — identify deteriorating bank performance
7. **PE PDF parsing** — extract structured data from Performance Evaluations
8. **Full reports** — Markdown summaries of CRA exam landscape
9. **Live FFIEC API** — search current ratings database

**Key use case:** Combined with `hmda-analyzer`, cra-scraper provides a complete
picture of bank community lending performance — HMDA shows mortgage activity,
CRA shows regulatory assessment of overall community reinvestment.

**Combined workflow:**
- `cra-scraper` finds banks with weak CRA ratings
- `hmda-analyzer` quantifies their mortgage lending disparities
- `cdfi-benchmark` compares them against MDI peers
- Together they create a complete fair lending and CRA compliance toolkit

**GitHub:** https://github.com/Jaypatel1511/cra-scraper
**PyPI:** https://pypi.org/project/cra-scraper
**Data:** https://www.ffiec.gov/craratings/
